# Detector Playground (Saved Model)

Loads trained artifacts from the `models/` folder, runs detection on CSV files from `test-logs/`, and saves outputs to `log-outputs/`.

In [2]:
import glob
from pathlib import Path
from collections import deque
import numpy as np
import pandas as pd
import joblib

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', None)

## 1. Load saved model artifacts

In [3]:
BASE_DIR = Path.cwd()
MODELS_DIR = BASE_DIR / 'models'
TEST_LOGS_DIR = BASE_DIR / 'test-logs'
OUTPUT_DIR = BASE_DIR / 'log-outputs'

OUTPUT_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
TEST_LOGS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / 'lgb_ssh_detector.pkl'
scaler_path = MODELS_DIR / 'scaler.pkl'
config_path = MODELS_DIR / 'detector_config.pkl'

assert model_path.exists(), f'Missing model file: {model_path}'
assert config_path.exists(), f'Missing config file: {config_path}'

saved_model = joblib.load(model_path)
saved_scaler = joblib.load(scaler_path) if scaler_path.exists() else None
saved_config = joblib.load(config_path)

best_thresh = float(saved_config['threshold'])
feat_cols = list(saved_config['feat_cols'])
WINDOW_SIZES = list(saved_config['window_sizes'])

print('Loaded artifacts from models/:')
print(f'  threshold: {best_thresh:.4f}')
print(f'  feature count: {len(feat_cols)}')
print(f'  windows: {WINDOW_SIZES}')

Loaded artifacts from models/:
  threshold: -0.0574
  feature count: 33
  windows: [30, 60, 300]


## 2. Detector class (same logic, using loaded artifacts)

In [4]:
class SSHBruteForceDetector:
    """
    Online SSH brute-force detector.
    Feed raw log lines one at a time via .ingest().
    Returns a score (0-1) and alert flag per line.
    """

    def __init__(self, model, scaler, threshold, feat_cols, window_sizes=[30, 60, 300]):
        self.model = model
        self.scaler = scaler
        self.threshold = threshold
        self.feat_cols = feat_cols
        self.window_sizes = window_sizes
        self.ip_buffers = {}
        self.max_window = max(window_sizes)

    def _get_buffer(self, ip):
        if ip not in self.ip_buffers:
            self.ip_buffers[ip] = deque()
        return self.ip_buffers[ip]

    def _prune(self, buf, now):
        while buf and (now - buf[0][0]) > self.max_window:
            buf.popleft()

    def _extract_features(self, buf, now):
        feats = {}
        for w in self.window_sizes:
            window = [e for e in buf if (now - e[0]) <= w]
            n = len(window)
            if n == 0:
                feats.update({f'w{w}_{k}': 0 for k in [
                    'n_attempts','attempt_rate','fail_ratio','unique_users',
                    'iat_mean','iat_std','iat_min','iat_cv',
                    'n_accepted','n_failed_pw','n_invalid'
                ]})
                continue

            statuses   = [e[1] for e in window]
            usernames  = [e[2] for e in window]
            events     = [e[3].lower() for e in window]
            timestamps = sorted([e[0] for e in window])

            n_failed  = sum(1 for s in statuses if s.lower() != 'success')
            fail_ratio = n_failed / n
            unique_users = len(set(usernames))
            attempt_rate = n / w

            iats = np.diff(timestamps) if len(timestamps) > 1 else np.array([0])
            iat_mean = iats.mean()
            iat_std  = iats.std()
            iat_min  = iats.min()
            iat_cv   = iat_std / iat_mean if iat_mean > 0 else 0

            n_accepted  = sum(1 for e in events if 'accept' in e)
            n_failed_pw = sum(1 for e in events if 'fail' in e)
            n_invalid   = sum(1 for e in events if 'invalid' in e)

            feats.update({
                f'w{w}_n_attempts':   n,
                f'w{w}_attempt_rate': attempt_rate,
                f'w{w}_fail_ratio':   fail_ratio,
                f'w{w}_unique_users': unique_users,
                f'w{w}_iat_mean':     iat_mean,
                f'w{w}_iat_std':      iat_std,
                f'w{w}_iat_min':      iat_min,
                f'w{w}_iat_cv':       iat_cv,
                f'w{w}_n_accepted':   n_accepted,
                f'w{w}_n_failed_pw':  n_failed_pw,
                f'w{w}_n_invalid':    n_invalid,
            })
        return feats

    def ingest(self, timestamp, source_ip, username, event_type, status):
        if isinstance(timestamp, str):
            ts = pd.Timestamp(timestamp).timestamp()
        elif isinstance(timestamp, pd.Timestamp):
            ts = timestamp.timestamp()
        else:
            ts = float(timestamp)

        buf = self._get_buffer(source_ip)
        self._prune(buf, ts)
        buf.append((ts, status, username, event_type))

        feats = self._extract_features(buf, ts)
        feat_vec_ordered = np.array([feats.get(col, 0) for col in self.feat_cols]).reshape(1, -1)

        if self.scaler is not None:
            model_input = feat_vec_ordered
        else:
            model_input = feat_vec_ordered

        score = float(self.model.predict_proba(model_input)[0, 1])
        alert = score >= self.threshold

        return {
            'source_ip': source_ip,
            'timestamp': timestamp,
            'score': round(score, 4),
            'alert': bool(alert),
            'features': feats
        }

    def process_csv(self, df_raw):
        required_cols = ['timestamp', 'source_ip', 'username', 'event_type', 'status']
        missing = [c for c in required_cols if c not in df_raw.columns]
        if missing:
            raise ValueError(f'Missing required columns: {missing}')

        results = []
        for idx, row in df_raw.iterrows():
            result = self.ingest(
                timestamp=row['timestamp'],
                source_ip=row['source_ip'],
                username=row['username'],
                event_type=row['event_type'],
                status=row['status']
            )
            results.append({
                'timestamp': result['timestamp'],
                'source_ip': result['source_ip'],
                'score': result['score'],
                'alert': result['alert'],
                'original_index': idx
            })
        return pd.DataFrame(results)

In [5]:
print("Load the model from the saved artificats")

detector = SSHBruteForceDetector(
    model=saved_model,
    scaler=saved_scaler,
    threshold=best_thresh,
    feat_cols=feat_cols,
    window_sizes=WINDOW_SIZES
)


Load the model from the saved artificats


## 3. Process logs from `test-logs/` and save to `log-outputs/`

In [6]:
input_files = sorted(glob.glob(str(TEST_LOGS_DIR / '*.csv')))
print(f'No. of log files in {TEST_LOGS_DIR}:  {len(input_files)} CSV file(s)')
for f in input_files:
    print(' -', Path(f).name)

No. of log files in /Users/amar/Desktop/Cybermac Course Materials/Second Sem/CCIP536 - ML/term project/ML-TImeSeries/test-logs:  4 CSV file(s)
 - log1.csv
 - log2.csv
 - log3.csv
 - log4.csv


In [ ]:
all_outputs = []

for file_path in input_files:
    in_path = Path(file_path)
    df_in = pd.read_csv(in_path)
    df_in['timestamp'] = pd.to_datetime(df_in['timestamp'])

    processed_results_df = detector.process_csv(df_in)

    final_display_df = df_in.merge(
        processed_results_df.drop(columns=['timestamp', 'source_ip']),
        left_index=True,
        right_on='original_index',
        how='left'
    ).drop(columns=['original_index'])

    out_name = f"{in_path.stem}_processed.csv"
    out_path = OUTPUT_DIR / out_name
    final_display_df.to_csv(out_path, index=False)

    print(f'Saved: {out_path}')
    print(f'  Alerts: {int(final_display_df["alert"].sum())} / {len(final_display_df)}')

    all_outputs.append((in_path.name, final_display_df))

if all_outputs:
    print('\nPreview of first processed file:')
    display(all_outputs[0][1].head(20))
else:
    print('No input files found. Put one or more CSV files in test-logs/.')